In [1]:
"""
@author: Zilan Cheng
@note: This code is modified based on Zongyi Li's original implementation of Fourier Neural Operators.
"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
torch.cuda.set_device(0)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
################################################################
#  1d fourier layer
################################################################
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1):
        super(SpectralConv1d, self).__init__()

        """
        1D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1

        self.scale = (1 / (in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul1d(self, input, weights):
        return torch.einsum("bix,iox->box", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft(x)

        out_ft = torch.zeros(batchsize, self.out_channels, x.size(-1)//2 + 1,  device=x.device, dtype=torch.cfloat)
        out_ft[:, :, :self.modes1] = self.compl_mul1d(x_ft[:, :, :self.modes1], self.weights1)
        x = torch.fft.irfft(out_ft, n=x.size(-1))
        return x

In [3]:
def get_grid(shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x))
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)

In [4]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv1d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv1d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [5]:
class FNO1d(nn.Module):
    def __init__(self, modes, width):
        super(FNO1d, self).__init__()

        self.modes1 = modes
        self.width = width
        self.padding = 8 # pad the domain if input is non-periodic

        self.p = nn.Linear(2, self.width) # input channel_dim is 2: (u0(x), x)
        self.conv0 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv1 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv2 = SpectralConv1d(self.width, self.width, self.modes1)
        self.conv3 = SpectralConv1d(self.width, self.width, self.modes1)
        self.mlp0 = MLP(self.width, self.width, self.width)
        self.mlp1 = MLP(self.width, self.width, self.width)
        self.mlp2 = MLP(self.width, self.width, self.width)
        self.mlp3 = MLP(self.width, self.width, self.width)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width*2)  # output channel_dim is 1: u1(x)

    def forward(self, x):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x=x.to(torch.float32)
        x = self.p(x)
        x = x.permute(0, 2, 1)
        x = F.pad(x, [0,self.padding]) # pad the domain if input is non-periodic

        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x = x1 + x2

        x = x[..., :-self.padding] # pad the domain if input is non-periodic
        x = self.q(x)
        x = x.permute(0, 2, 1)
        return x

In [6]:
################################################################
#  configurations
################################################################
ntrain = 900
ntest = 100

batch_size = 20
learning_rate = 0.001
epochs = 1000
iterations = epochs*(ntrain//batch_size)

modes = 8
width = 32

size=1024

In [7]:
################################################################
# dataloader
################################################################
data = np.load("../data/sum_sin/sine_data.npz", allow_pickle=True)

X = data["input"]  
Y = data["output"]

x_train = X[:ntrain,  :] 
y_train = Y[:ntrain, :] 

x_test  = X[-ntest:, :]
y_test  = Y[-ntest:, :]

x_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)

x_test  = torch.tensor(x_test,  dtype=torch.float32).unsqueeze(-1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(-1)

x_train = x_train.reshape(ntrain,size,1)
x_test = x_test.reshape(ntest,size,1)

y_train = y_train.reshape(ntrain,size,1)
y_test = y_test.reshape(ntest,size,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

In [8]:
################################################################
# training and evaluation
################################################################

model = FNO1d(modes, width)
model=model.to(device)
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)
myloss = LpLoss(size_average=False)
y_normalizer.to(device)
for ep in range(epochs):
    model.train()
    train_mse = 0
    train_l2 = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
    
        optimizer.zero_grad()
        out = model(x)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)
        l2 = myloss(out.view(batch_size, -1), y.view(batch_size, -1))
        l2.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += l2.item()
    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            out = y_normalizer.decode(out)
            test_l2 += myloss(out.view(batch_size, -1), y.view(batch_size, -1)).item()

    train_l2/= ntrain
    test_l2 /= ntest

    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Train L2: {train_l2:.6f} | Test L2: {test_l2:.6f}")

80481
Epoch    0 | Train L2: 0.947550 | Test L2: 0.860480
Epoch   50 | Train L2: 0.514114 | Test L2: 0.535149
Epoch  100 | Train L2: 0.478448 | Test L2: 0.505093
Epoch  150 | Train L2: 0.462029 | Test L2: 0.503423
Epoch  200 | Train L2: 0.457215 | Test L2: 0.484738
Epoch  250 | Train L2: 0.446229 | Test L2: 0.475901
Epoch  300 | Train L2: 0.440847 | Test L2: 0.469926
Epoch  350 | Train L2: 0.427205 | Test L2: 0.457582
Epoch  400 | Train L2: 0.407481 | Test L2: 0.450575
Epoch  450 | Train L2: 0.392376 | Test L2: 0.424503
Epoch  500 | Train L2: 0.381514 | Test L2: 0.411799
Epoch  550 | Train L2: 0.364820 | Test L2: 0.399885
Epoch  600 | Train L2: 0.346027 | Test L2: 0.382755
Epoch  650 | Train L2: 0.335444 | Test L2: 0.370444
Epoch  700 | Train L2: 0.325397 | Test L2: 0.359428
Epoch  750 | Train L2: 0.320876 | Test L2: 0.353907
Epoch  800 | Train L2: 0.311859 | Test L2: 0.346082
Epoch  850 | Train L2: 0.307741 | Test L2: 0.342443
Epoch  900 | Train L2: 0.304773 | Test L2: 0.339431
Epoch 